# Text-to-SQL PT-BR · QLoRA fine-tuning on Colab T4

End-to-end notebook: generates the synthetic dataset, fine-tunes Qwen2.5-0.5B with QLoRA, runs an ablation (rank 8 vs rank 16), evaluates on a held-out test split, plots loss curves, and pushes the adapter to the Hugging Face Hub.

**Requirements**: Colab **T4 GPU** (free tier works). `Runtime → Change runtime type → T4 GPU`.

**Expected wall-clock**: ~45 min for base + rank8 ablation.

## 0 · Sanity check GPU

In [ ]:
!nvidia-smi

## 1 · Clone repo and install deps

In [ ]:
# Replace with your repo URL after pushing to GitHub:
!git clone https://github.com/baconshow/text2sql-ptbr-lora.git
%cd text2sql-ptbr-lora
!pip install -q -r requirements.txt

## 2 · Generate synthetic dataset (3k examples)

In [ ]:
!python src/data_prep.py --output data --n_samples 3000 --seed 42

## 3 · Train the base run (LoRA r=16)

In [ ]:
!python src/train.py --config configs/base.yaml

## 4 · Ablation — smaller rank (r=8)

Half the trainable params. Does it hurt accuracy?

In [ ]:
!python src/train.py --config configs/ablation_rank8.yaml

## 5 · Evaluate both runs on the held-out test set

In [ ]:
!python src/evaluate.py --adapter runs/base --data data/test.jsonl --base_model Qwen/Qwen2.5-0.5B-Instruct
!python src/evaluate.py --adapter runs/rank8 --data data/test.jsonl --base_model Qwen/Qwen2.5-0.5B-Instruct

## 6 · Baseline (no fine-tune) for comparison

In [ ]:
# Evaluate the base model WITHOUT the LoRA adapter applied:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.evaluate import generate, normalize, token_f1

base = 'Qwen/Qwen2.5-0.5B-Instruct'
tok = AutoTokenizer.from_pretrained(base, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
m = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True)
m.eval()

import pathlib
exs = [json.loads(l) for l in open('data/test.jsonl')][:100]  # 100 is enough for baseline
em=0; f1=0.0
for ex in exs:
    pred = generate(m, tok, ex['messages'])
    gold = ex['messages'][-1]['content']
    em += int(normalize(pred)==normalize(gold))
    f1 += token_f1(pred, gold)
print({'baseline_em': em/len(exs), 'baseline_f1': f1/len(exs)})
json.dump({'em': em/len(exs), 'f1': f1/len(exs)}, open('runs/baseline_metrics.json','w'))

## 7 · Plot loss curves

In [ ]:
import json, matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,5))
for run in ['base', 'rank8']:
    logs = json.load(open(f'runs/{run}/log_history.json'))
    steps = [e['step'] for e in logs if 'loss' in e]
    losses = [e['loss'] for e in logs if 'loss' in e]
    ax.plot(steps, losses, label=run)
ax.set_xlabel('Step'); ax.set_ylabel('Train loss'); ax.legend(); ax.grid(alpha=.3)
ax.set_title('QLoRA training loss — ablation')
plt.savefig('assets/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 · Build the ablation markdown table

In [ ]:
import json
rows = []
baseline = json.load(open('runs/baseline_metrics.json'))
rows.append(('Qwen2.5-0.5B (no fine-tune)', '-', '-', f"{baseline['em']:.3f}", f"{baseline['f1']:.3f}"))
for run in ['base', 'rank8']:
    cfg = json.load(open(f'runs/{run}/metrics.json'))['config']
    ev  = json.load(open(f'runs/{run}/eval_results.json'))['summary']
    rows.append((run, cfg['lora_r'], cfg['learning_rate'], f"{ev['exact_match']:.3f}", f"{ev['token_f1']:.3f}"))

md = '| Run | LoRA r | LR | Exact Match | Token F1 |\n|---|---|---|---|---|\n'
for r in rows:
    md += '| ' + ' | '.join(str(x) for x in r) + ' |\n'
print(md)
open('results/ablation_table.md','w').write(md)

## 9 · Push the best adapter to the Hugging Face Hub

Create a token at https://huggingface.co/settings/tokens (WRITE scope).

In [ ]:
from huggingface_hub import login, create_repo, upload_folder
import getpass
login(token=getpass.getpass('HF token: '))

repo_id = 'baconshow/qwen2.5-0.5b-text2sql-ptbr-lora'  # change if you want
create_repo(repo_id, exist_ok=True)
upload_folder(folder_path='runs/base', repo_id=repo_id, commit_message='Add LoRA adapter (base run)')
print(f'Published: https://huggingface.co/{repo_id}')